# A5 · Baselines: the score to beat

**Where we are**

```
0 Frame    → what we predict: a SAFE slot length (the P-point), set by the PO's late-vs-idle ratio
1 Collect  → simulator made the visits
2 Explore  → found 8 problems
3 Prepare  → fixed them; split by date: train Jan–May, valid June, test Jul–Aug
4 Baseline → HERE: simple predictions with no ML, scored on June
5 Train    → LightGBM must beat this score (A6)
```

**Words for this lesson**

| Word | Plain meaning | Example |
|---|---|---|
| actual (`y`) | how long the visit really took: the `minutes` column | 25 |
| prediction (`p`) | the predicted P-point for that visit | 16.6 |
| `q` | the P value as a fraction: 0.67, 0.75 or 0.80. A **business setting** from the PO (A1), not something we choose. We compute everything for all three; the PO picks in A8 | 0.8 |
| late | minutes the visit ran past its prediction: `max(y − p, 0)` | 25 − 20 = 5 |
| idle | minutes the doctor sat free: `max(p − y, 0)` | 20 − 15 = 5 |
| score (pinball loss) | per visit: `late × q + idle × (1 − q)`; then averaged over all visits. **Lower is better** | |
| coverage | share of visits with `y ≤ p`; for a good P80 it's about 0.80 | |
| slot | what gets booked: `p` rounded **up** to 5 minutes | 16.6 → 20 |

**Before running this notebook,** write the `TODO(you)` parts and make the tests pass:

| File | Your parts | Test |
|---|---|---|
| `slotpred/metrics.py` | `pinball_loss`, `coverage`, `to_slot`, `booked_metrics` | `.venv/Scripts/python.exe -m pytest tests/test_metrics.py -v` |
| `slotpred/baselines.py` | `GroupQuantile.fit`, `GroupQuantile.predict` | `.venv/Scripts/python.exe -m pytest tests/test_baselines.py -v` |

**Rule for this lesson:** everything is fitted on **train** and scored on **valid (June)**. The test months stay locked until A8.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from slotpred.baselines import FixedSlot, GlobalQuantile, GroupQuantile, as_frame, oracle_quantile
from slotpred.config import DATA, OUT, Q_CANDIDATES
from slotpred.features import add_time_features
from slotpred.metrics import booked_metrics, coverage, evaluate, pinball_loss, to_slot
from slotpred.prepare import load_prepared
from slotpred.world import NOISE_SIGMA

splits, state = load_prepared()
train, valid = splits["train"], splits["valid"]
y = valid["minutes"].to_numpy()          # the June actuals every method is scored against
print(len(train), "training visits,", len(y), "June visits")
print("P values (from the PO, A1):", Q_CANDIDATES)

## 1 · Check your score functions on the quick-check visits

Slot 20, q = 0.8. A 30-minute visit is 10 minutes late → 10 × 0.8 = **8.0**. A 12-minute visit leaves 8 idle minutes → 8 × 0.2 = **1.6**. Average = **4.8**.

In [ ]:
print(pinball_loss([30, 12], [20, 20], 0.8))       # expect 4.8
print(coverage([30, 12], [20, 20]))                 # expect 0.5: one of two finished in time
print(to_slot(np.array([16.6, 15.0, 2.0])))         # expect [20. 15.  5.]
print(booked_metrics(np.array([30, 12]), np.array([20, 20])))

## 2 · The four baselines, and the best possible score

| | Method | Learns from train |
|---|---|---|
| B0 | `FixedSlot(15)`: today's rule | nothing |
| B1 | `GlobalQuantile(q)`: one number for everyone | 1 number |
| B2 | `GroupQuantile(["department", "visit_type"], q)`: one number per box | 24 numbers |
| B3 | `GroupQuantile(["doctor", "visit_type"], q, min_n=30, parent=B2)`: smaller boxes, backoff to B2 | 60 numbers |

**The oracle** is not a method. It uses the simulator's hidden `expected_minutes` (A2) to compute the best possible P-point for each visit. No real model can see that column; we use it **only to know the floor**, the best score any model could reach. Only the random luck is left.

In [ ]:
truth = pd.read_csv(DATA / "_hidden_truth.csv")                       # for SCORING only, never a feature
expected = valid[["appointment_id"]].merge(truth, on="appointment_id", how="left")["expected_minutes"].to_numpy()

def baselines(q):
    return {
        "B0 fixed 15": FixedSlot(15),
        "B1 one number": GlobalQuantile(q),
        "B2 department + visit type": GroupQuantile(["department", "visit_type"], q),
        "B3 doctor + visit type (backoff to B2)": GroupQuantile(["doctor", "visit_type"], q, min_n=30,
                                                                  parent=GroupQuantile(["department", "visit_type"], q)),
    }

In [ ]:
# TODO(you): build the June leaderboard for every q in Q_CANDIDATES.
# For each q:
#   for each (name, method) in baselines(q).items():
#       method.fit(train); pred = method.predict(valid); add evaluate(y, pred, q, name) to a list
#   also add evaluate(y, oracle_quantile(expected, q, NOISE_SIGMA), q, "Oracle (floor)")
# Then: board = as_frame(rows); display it.

**Questions** (look at the q = 0.8 rows first)
1. Which baseline has the lowest score? How far is it from the oracle?
2. B1's coverage is about right (close to q), yet its score is bad. How can both be true? *(Hint: quick-check question 2.)*
3. B0 books 15 for everyone. Which q does its coverage look like? Is "15 minutes" secretly a P-something?

*Answers:*

## 3 · How small can a box be?

Smaller boxes are more specific, but each holds fewer visits, so its P-point is shakier (the restaurant-reviews idea). Try finer and finer boxes at q = 0.8, first **without** backoff (`min_n=1`, every box is trusted), then **with** backoff (`min_n=30`, small boxes use B2).

`hour` and `weekday` aren't columns yet; `add_time_features` (A4) adds them.

In [ ]:
q = 0.8
train_t, valid_t = add_time_features(train), add_time_features(valid)
ladder = [
    ["department", "visit_type"],
    ["doctor", "visit_type"],
    ["doctor", "visit_type", "clinic"],
    ["doctor", "visit_type", "clinic", "weekday"],
    ["doctor", "visit_type", "clinic", "weekday", "hour"],
]
# TODO(you): for each keys in ladder, and for min_n in (1, 30):
#   model = GroupQuantile(keys, q, min_n=min_n, parent=GroupQuantile(["department", "visit_type"], q))
#   fit on train_t, predict valid_t, and record: number of boxes (len(model.table_)),
#   the median visits per box (model.counts_.median()), min_n and pinball_loss(y, pred, q)

**Questions**
1. At which box size does the score stop improving and start getting worse without backoff?
2. What does backoff do to the finest row? Why can't it make things *worse* than B2?

*Answers:*

## 4 · Rounding up to the 5-minute grid

The model predicts a P-point like 16.6, but the scheduler books **20**. That extra margin means more visits fit than q suggests. Compare, for B2 at each q, the **coverage** (on the raw prediction) with **booked_on_time** (on the booked slot).

In [ ]:
# TODO(you): from your leaderboard, show method, q, coverage, booked_on_time, booked_late_min, booked_idle_min
# for the B2 rows only. hint: board[board.method.str.startswith("B2")][[...]]

**Questions**
1. For q = 0.8, coverage is about 0.79 but booked on-time is higher. By how much, and why?
2. If the PO wants about 80% of visits on time **as booked**, which q gets closest? Who decides in the end?

*Answers:*

## 5 · Save the leaderboard for A6

LightGBM will be compared against these exact numbers.

In [ ]:
OUT.mkdir(exist_ok=True)
board.to_csv(OUT / "baselines_valid.csv", index=False)
print("saved", OUT / "baselines_valid.csv")

## Wrap-up (answer in `notes/A05-baselines.md` too, in your own words)

1. What is a baseline for? Give its three jobs.
2. Why is the score `late × q + idle × (1 − q)`, and where does q come from?
3. Why not make the boxes as small as possible, and what does backoff do?
4. LightGBM in A6 also builds boxes. What will it do differently from B2?